In [ ]:
!pip install -q transformers datasets sentencepiece accelerate
!pip install -q scikit-learn scipy pandas tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig
)

from datasets import load_dataset

from scipy.stats import spearmanr
from scipy.stats import pearsonr

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from tqdm import tqdm

import random
import numpy as np

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


Load BERT

In [ ]:
MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

encoder = AutoModel.from_pretrained(
    MODEL_NAME
)

encoder.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

Load SNLI Dataset

In [ ]:
from datasets import load_dataset

snli = load_dataset(
    "stanfordnlp/snli",
    split="train"
)

print(snli)

README.md:   0%|          | 0.00/16.0k [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/412k [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/413k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/19.6M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/550152 [00:00<?, ? examples/s]

Dataset({
    features: ['premise', 'hypothesis', 'label'],
    num_rows: 550152
})


In [ ]:
print(snli[0])

{'premise': 'A person on a horse jumps over a broken down airplane.', 'hypothesis': 'A person is training his horse for a competition.', 'label': 1}


Prepare Training Sentences

In [ ]:
train_sentences = []

for i in range(len(snli)):

    premise = snli[i]["premise"]
    hypothesis = snli[i]["hypothesis"]

    if premise is not None and len(premise.strip()) > 0:
        train_sentences.append(premise)

    if hypothesis is not None and len(hypothesis.strip()) > 0:
        train_sentences.append(hypothesis)

print("Total Sentences:", len(train_sentences))

print("\nExample Sentences:")
for i in range(5):
    print(train_sentences[i])

Total Sentences: 1100304

Example Sentences:
A person on a horse jumps over a broken down airplane.
A person is training his horse for a competition.
A person on a horse jumps over a broken down airplane.
A person is at a diner, ordering an omelette.
A person on a horse jumps over a broken down airplane.


Corruption Function:

To create positive pairs for contrastive learning, a corrupted version of each sentence is generated.
A random masking strategy is used: mask_ratio = 0.30


In [ ]:
import random

def corrupt_sentence(sentence, mask_ratio=0.30):

    tokens = tokenizer.tokenize(sentence)

    corrupted_tokens = []

    for token in tokens:

        if random.random() < mask_ratio:
            corrupted_tokens.append("[MASK]")
        else:
            corrupted_tokens.append(token)

    return tokenizer.convert_tokens_to_string(
        corrupted_tokens
    )

In [ ]:
sample = train_sentences[0]

print("Original:")
print(sample)

print("\nCorrupted:")
print(corrupt_sentence(sample))

Original:
A person on a horse jumps over a broken down airplane.

Corrupted:
a [MASK] [MASK] [MASK] horse jumps over [MASK] broken [MASK] [MASK].


Creating Dataset Class:
For every sentence, the dataset returns:Original Sentence and Corrupted Sentence


In [ ]:
from torch.utils.data import Dataset

class DiffCSEDataset(Dataset):

    def __init__(self, sentences):

        self.sentences = sentences

    def __len__(self):

        return len(self.sentences)

    def __getitem__(self, idx):

        original = self.sentences[idx]

        corrupted = corrupt_sentence(
            original
        )

        return {
            "original": original,
            "corrupted": corrupted
        }

Creating Dataset

In [ ]:
dataset = DiffCSEDataset(
    train_sentences[:20000]
)

print(len(dataset))

20000


DataLoader

In [ ]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)

print(len(loader))

313


**Mean Pooling**:
The mean pooling function converts token-level BERT embeddings into a single sentence embedding. It averages the embeddings of all valid tokens while ignoring padding tokens through the attention mask. This produces one fixed-length vector representation for each sentence, which is later used for contrastive learning and similarity computation.

In [ ]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    mask = attention_mask.unsqueeze(-1)

    mask = mask.expand(
        token_embeddings.size()
    ).float()

    summed = torch.sum(
        token_embeddings * mask,
        dim=1
    )

    counts = torch.clamp(
        mask.sum(dim=1),
        min=1e-9
    )

    return summed / counts

Sentence Encoder:

Sentence -> Tokenizer -> Token IDs -> BERT Encoder -> Token Embeddings -> Mean Pooling -> Sentence Embedding -> L2 Normalization -> Final Embedding

In [ ]:
import torch.nn.functional as F

def encode_sentences(sentences):

    inputs = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    outputs = encoder(**inputs)

    embeddings = mean_pooling(
        outputs,
        inputs["attention_mask"]
    )
"""L2 normalization divides a vector by its Euclidean length so that the resulting vector has a magnitude of 1.
It preserves the direction of the vector and is commonly used before cosine similarity calculations."""
    embeddings = F.normalize(
        embeddings,
        p=2,
        dim=1
    )

    return embeddings

Testing Encoder

In [ ]:
emb = encode_sentences(
    train_sentences[:4]
)

print(emb.shape)

torch.Size([4, 768])


Defining Contrastive Loss:

For each batch: Original Sentence -> Embedding z₁
Corrupted Sentence -> Embedding z₂

Similarity is computed using matrix multiplication and scaled by a temperature parameter: temperature = 0.05

Cross-entropy loss is then used to optimize the embeddings. Through backpropagation, it generates gradients that increase the similarity of positive pairs and decrease the similarity of negative pairs, causing the encoder to produce better sentence embeddings.


In [ ]:
import torch.nn as nn

class InfoNCELoss(nn.Module):

    def __init__(
            self,
            temperature=0.05
    ):
        super().__init__()

        self.temperature = temperature

    def forward(
            self,
            z1,
            z2
    ):

        similarity = torch.matmul(
            z1,
            z2.T
        )

        similarity /= self.temperature

        labels = torch.arange(
            z1.size(0)
        ).to(device)

        loss = F.cross_entropy(
            similarity,
            labels
        )

        return loss

Integrating Optimzer

In [ ]:
criterion = InfoNCELoss()

optimizer = torch.optim.AdamW(
    #AdamW is improved version of Adam that includes weight decay regularization.
    #Fast convergence and stable training
    encoder.parameters(),
    lr=7e-6
)

Training

In [ ]:
EPOCHS = 2

encoder.train()

for epoch in range(EPOCHS):

    total_loss = 0

    for batch in loader:

        originals = batch["original"]
        corrupteds = batch["corrupted"]

        emb1 = encode_sentences(
            originals
        )

        emb2 = encode_sentences(
            corrupteds
        )

        loss = criterion(
            emb1,
            emb2
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)

    print(
        f"Epoch {epoch+1} Loss = {avg_loss:.4f}"
    )

Epoch 1 Loss = 0.1819
Epoch 2 Loss = 0.0443


In [ ]:
torch.save(
    encoder.state_dict(),
    "diffcse_style_model.pt"
)

print("Model saved successfully.")

Model saved successfully.


Load STS-B

In [ ]:
from datasets import load_dataset

ds = load_dataset("sentence-transformers/stsb")

README.md:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/471k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/142k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/108k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

In [ ]:
print(ds)
print(ds["test"][0])

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'score'],
        num_rows: 1379
    })
})
{'sentence1': 'A girl is styling her hair.', 'sentence2': 'A girl is brushing her hair.', 'score': 0.5}


Evaluation

Pearson and Spearman correlation coefficients were used to evaluate the quality of the learned sentence embeddings on the STS benchmark. Pearson correlation measures the linear relationship between model-predicted similarity scores and human-annotated scores, while Spearman correlation measures how well the ranking of sentence similarities matches human judgments. Higher correlation values indicate better semantic representations.

In [ ]:
from scipy.stats import pearsonr, spearmanr
from tqdm import tqdm
import torch
import torch.nn.functional as F

test_data = ds["test"]

predictions = []
gold_scores = []

batch_size = 64

encoder.eval()

with torch.no_grad():

    for start in tqdm(
        range(0, len(test_data), batch_size)
    ):

        batch = test_data[
            start:start+batch_size
        ]

        s1 = batch["sentence1"]
        s2 = batch["sentence2"]

        emb1 = encode_sentences(s1)
        emb2 = encode_sentences(s2)

        similarities = F.cosine_similarity(
            emb1,
            emb2
        )

        predictions.extend(
            similarities.cpu().numpy().tolist()
        )

        gold_scores.extend(
            batch["score"]
        )

pearson = pearsonr(
    predictions,
    gold_scores
)[0]

spearman = spearmanr(
    predictions,
    gold_scores
)[0]

print(f"Pearson Correlation : {pearson:.4f}")
print(f"Spearman Correlation: {spearman:.4f}")

100%|██████████| 22/22 [00:04<00:00,  5.30it/s]

Pearson Correlation : 0.6927
Spearman Correlation: 0.6768


In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["DiffCSE-style Reimplementation"],
    "Pearson": [round(pearson,4)],
    "Spearman": [round(spearman,4)]
})

results

,Model,Pearson,Spearman
0,DiffCSE-style Reimplementation,0.6927,0.6768


Running the repo

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(
    "voidism/diffcse-bert-base-uncased-sts"
)

model = AutoModel.from_pretrained(
    "voidism/diffcse-bert-base-uncased-sts"
)

print("Loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.25G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: voidism/diffcse-bert-base-uncased-sts
Key                                                                                | Status     | 
-----------------------------------------------------------------------------------+------------+-
generator.distilbert.transformer.layer.{0, 1, 2, 3, 4, 5}.attention.out_lin.weight | UNEXPECTED | 
aux_bert.encoder.layer.{0...11}.output.dense.bias                                  | UNEXPECTED | 
aux_bert.encoder.layer.{0...11}.intermediate.dense.weight                          | UNEXPECTED | 
aux_bert.encoder.layer.{0...11}.intermediate.dense.bias                            | UNEXPECTED | 
aux_bert.encoder.layer.{0...11}.output.LayerNorm.bias                              | UNEXPECTED | 
generator.distilbert.embeddings.LayerNorm.weight                                   | UNEXPECTED | 
aux_bert.encoder.layer.{0...11}.attention.self.value.bias                          | UNEXPECTED | 
aux_bert.encoder.layer.{0...11}.attention.s

Loaded successfully!


In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print(device)

cuda


In [ ]:
import torch
import torch.nn.functional as F

def get_embeddings(sentences):

    inputs = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model(**inputs)

        embeddings = outputs.last_hidden_state[:, 0]

        embeddings = F.normalize(
            embeddings,
            p=2,
            dim=1
        )

    return embeddings

In [ ]:
emb = get_embeddings([
    "A cat is sleeping.",
    "A dog is running."
])

print(emb.shape)

torch.Size([2, 768])


In [ ]:
from datasets import load_dataset

ds = load_dataset("sentence-transformers/stsb")

README.md:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/471k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/142k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/108k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

In [ ]:
test_data = ds["test"]

In [ ]:
from tqdm import tqdm
from scipy.stats import pearsonr, spearmanr

predictions = []
gold_scores = []

batch_size = 64

for start in tqdm(
    range(0, len(test_data), batch_size)
):

    batch = test_data[
        start:start+batch_size
    ]

    s1 = batch["sentence1"]
    s2 = batch["sentence2"]

    emb1 = get_embeddings(s1)
    emb2 = get_embeddings(s2)

    sims = F.cosine_similarity(
        emb1,
        emb2
    )

    predictions.extend(
        sims.cpu().numpy().tolist()
    )

    gold_scores.extend(
        batch["score"]
    )

pearson = pearsonr(
    predictions,
    gold_scores
)[0]

spearman = spearmanr(
    predictions,
    gold_scores
)[0]

print(
    f"Pearson  = {pearson:.4f}"
)

print(
    f"Spearman = {spearman:.4f}"
)

100%|██████████| 22/22 [00:04<00:00,  5.38it/s]

Pearson  = 0.6884
Spearman = 0.6979
